In [23]:
import pandas as pd

mod_df = pd.read_json("../Data/modcloth_final_data.json", lines=True)
print(mod_df.shape[0])
mod_df.head()

82790


,item_id,waist,size,quality,cup size,hips,bra size,category,bust,height,user_name,length,fit,user_id,shoe size,shoe width,review_summary,review_text
0,123373,29.0,7,5.0,d,38.0,34.0,new,36,5ft 6in,Emily,just right,small,991571,NaN,NaN,NaN,NaN
1,123373,31.0,13,3.0,b,30.0,36.0,new,NaN,5ft 2in,sydneybraden2001,just right,small,587883,NaN,NaN,NaN,NaN
2,123373,30.0,7,2.0,b,NaN,32.0,new,NaN,5ft 7in,Ugggh,slightly long,small,395665,9.0,NaN,NaN,NaN
3,123373,NaN,21,5.0,dd/e,NaN,NaN,new,NaN,NaN,alexmeyer626,just right,fit,875643,NaN,NaN,NaN,NaN
4,123373,NaN,18,5.0,b,NaN,36.0,new,NaN,5ft 2in,dberrones1,slightly long,small,944840,NaN,NaN,NaN,NaN


In [8]:
sizes_sorted = sorted(mod_df["size"].dropna().unique())
print("Unique values for size (ascending):")
print(sizes_sorted)

Unique values for size (ascending):
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(17), np.int64(18), np.int64(20), np.int64(21), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(38)]


In [9]:
quality_sorted = sorted(mod_df["quality"].dropna().unique())
print("Unique values for quality:")
print(quality_sorted)

Unique values for quality:
[np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0)]


In [10]:
length_sorted = sorted(mod_df["length"].dropna().unique())
print("Unique values for length:")
print(length_sorted)

Unique values for length:
['just right', 'slightly long', 'slightly short', 'very long', 'very short']


In [11]:
fit_sorted = sorted(mod_df["fit"].dropna().unique())
print("Unique values for fit:")
print(fit_sorted)

Unique values for fit:
['fit', 'large', 'small']


In [12]:
category_sorted = sorted(mod_df["category"].dropna().unique())
print("Unique values for category:")
print(category_sorted)

Unique values for category:
['bottoms', 'dresses', 'new', 'outerwear', 'sale', 'tops', 'wedding']


In [13]:
print("Total null values for each column:")
print(mod_df.isnull().sum())

Total null values for each column:
item_id               0
waist             79908
size                  0
quality              68
cup size           6255
hips              26726
bra size           6018
category              0
bust              70936
height             1107
user_name             0
length               35
fit                   0
user_id               0
shoe size         54875
shoe width        64183
review_summary     6725
review_text        6725
dtype: int64


In [31]:
distribution = mod_df.groupby("category").agg(
    total_rows=("category", "size"),
    bra_size_not_null=("bra size", "count"),
    cup_size_not_null=("cup size", "count"),
    waist_not_null=("waist", "count"),
    hips_not_null=("hips", "count")
)

distribution["bra_size_not_null_pct"] = (
    distribution["bra_size_not_null"] / distribution["total_rows"] * 100
).round(2)
distribution["cup_size_not_null_pct"] = (
    distribution["cup_size_not_null"] / distribution["total_rows"] * 100
).round(2)
distribution["waist_not_null_pct"] = (
    distribution["waist_not_null"] / distribution["total_rows"] * 100
).round(2)
distribution["hips_not_null_pct"] = (
    distribution["hips_not_null"] / distribution["total_rows"] * 100
).round(2)

print("Category-wise distribution of non-null bra size, cup size, waist, and hips rows:")
distribution.sort_values("total_rows", ascending=False)

Category-wise distribution of non-null bra size, cup size, waist, and hips rows:


,total_rows,bra_size_not_null,cup_size_not_null,waist_not_null,hips_not_null,bra_size_not_null_pct,cup_size_not_null_pct,waist_not_null_pct,hips_not_null_pct
category,,,,,,,,,
new,21488,20119,20037,316,14518,93.63,93.25,1.47,67.56
tops,20364,19144,19106,653,13694,94.01,93.82,3.21,67.25
dresses,18650,17475,17443,901,12290,93.70,93.53,4.83,65.90
bottoms,15266,13561,13503,451,10961,88.83,88.45,2.95,71.80
outerwear,4223,3966,3955,94,2839,93.91,93.65,2.23,67.23
sale,2524,2262,2245,463,1604,89.62,88.95,18.34,63.55
wedding,275,245,246,4,158,89.09,89.45,1.45,57.45


In [33]:
target_categories = ["tops", "dresses"]
category_norm = mod_df["category"].astype(str).str.strip().str.lower().replace({"dress": "dresses"})

counts = (
    category_norm[
        category_norm.isin(target_categories)
        & mod_df["cup size"].notna()
        & mod_df["bra size"].notna()
        & mod_df["hips"].notna()
    ]
    .value_counts()
    .reindex(target_categories, fill_value=0)
    .astype(int)
)

print("Counts where cup size, bra size, and hips are not null:")
print(counts)
print(f"Total: {counts.sum()}")

Counts where cup size, bra size, and hips are not null:
category
tops       13449
dresses    12027
Name: count, dtype: int64
Total: 25476


In [35]:
tops_dresses_df = mod_df[
    mod_df["category"].astype(str).str.strip().str.lower().isin(["tops", "dresses"])
] .copy()

columns_to_drop = [
    "item_id",
    "waist",
    "quality",
    "bust",
    "user_name",
    "length",
    "user_id",
    "show_size",
    "shoe_width",
    "review_summary",
    "review_text",
    "shoe size",
    "shoe width",
]

tops_dresses_df = tops_dresses_df.drop(
    columns=[col for col in columns_to_drop if col in tops_dresses_df.columns]
 )

tops_dresses_df = tops_dresses_df.dropna().reset_index(drop=True)

print("New dataframe created: tops and dresses only")
print("Shape after dropping rows with any null values:", tops_dresses_df.shape)
tops_dresses_df.head()

New dataframe created: tops and dresses only
Shape after dropping rows with any null values: (25421, 7)


,size,cup size,hips,bra size,category,height,fit
0,8,c,35.0,34.0,dresses,5ft 4in,small
1,1,a,38.0,32.0,dresses,5ft 4in,fit
2,8,d,41.0,36.0,dresses,5ft 2in,fit
3,12,dd/e,36.0,34.0,dresses,5ft 5in,fit
4,4,c,35.0,34.0,dresses,5ft 4in,fit


In [38]:
cup_series = tops_dresses_df["cup size"].astype(str).str.strip().str.lower()
cup_values = sorted(cup_series.unique())
cup_map = {label: idx for idx, label in enumerate(cup_values)}
tops_dresses_df["cup size"] = cup_series.map(cup_map).astype("Int64")

height_parts = tops_dresses_df["height"].astype(str).str.lower().str.extract(
    r"^\s*(?P<feet>\d+)\s*ft(?:\s*(?P<inches>\d+)\s*in)?\s*$"
 )
tops_dresses_df["height"] = (
    pd.to_numeric(height_parts["feet"], errors="coerce") * 12
    + pd.to_numeric(height_parts["inches"], errors="coerce").fillna(0)
).astype("Int64")

fit_map = {"small": 0, "fit": 1, "large": 2}
tops_dresses_df["fit"] = (
    tops_dresses_df["fit"].astype(str).str.strip().str.lower().map(fit_map).astype("Int64")
)

category_map = {"tops": 0, "dresses": 1}
tops_dresses_df["category"] = (
    tops_dresses_df["category"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map(category_map)
    .astype("Int64")
)

print("Cup size mapping:", cup_map)
print("Fit mapping:", fit_map)
print("Category mapping:", category_map)
print("Nulls after encoding:")
print(tops_dresses_df[["cup size", "height", "fit", "category"]].isna().sum())
tops_dresses_df.head()

Cup size mapping: {'a': 0, 'aa': 1, 'b': 2, 'c': 3, 'd': 4, 'dd/e': 5, 'ddd/f': 6, 'dddd/g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11}
Fit mapping: {'small': 0, 'fit': 1, 'large': 2}
Category mapping: {'tops': 0, 'dresses': 1}
Nulls after encoding:
cup size    0
height      0
fit         0
category    0
dtype: int64


,size,cup size,hips,bra size,category,height,fit
0,8,3,35.0,34.0,1,64,0
1,1,0,38.0,32.0,1,64,1
2,8,4,41.0,36.0,1,62,1
3,12,5,36.0,34.0,1,65,1
4,4,3,35.0,34.0,1,64,1


In [41]:
output_path = "../Data/cleaned_data.csv"
tops_dresses_df.to_csv(output_path, index=False)
print(f"Saved cleaned data to {output_path}")
print("Rows:", tops_dresses_df.shape[0])
print("Columns:", tops_dresses_df.shape[1])

Saved cleaned data to ../Data/cleaned_data.csv
Rows: 25421
Columns: 7


In [42]:
split_seed = 42
train_fraction = 0.8

train_parts = []
for category_value, group in tops_dresses_df.groupby("category"):
    n_total = len(group)
    n_train = int(round(n_total * train_fraction))
    if n_total > 1:
        n_train = min(max(n_train, 1), n_total - 1)
    train_parts.append(group.sample(n=n_train, random_state=split_seed))

train_df = pd.concat(train_parts)
test_df = tops_dresses_df.drop(index=train_df.index)

train_df = train_df.sample(frac=1, random_state=split_seed).reset_index(drop=True)
test_df = test_df.sample(frac=1, random_state=split_seed).reset_index(drop=True)

train_path = "../Data/train.csv"
test_path = "../Data/test.csv"

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

split_stats = pd.DataFrame({
    "train_rows": train_df["category"].value_counts(),
    "test_rows": test_df["category"].value_counts(),
}).fillna(0).astype(int)
split_stats["total_rows"] = split_stats["train_rows"] + split_stats["test_rows"]
split_stats["train_pct"] = (split_stats["train_rows"] / split_stats["total_rows"] * 100).round(2)
split_stats["test_pct"] = (split_stats["test_rows"] / split_stats["total_rows"] * 100).round(2)

print(f"Saved: {train_path} ({train_df.shape[0]} rows)")
print(f"Saved: {test_path} ({test_df.shape[0]} rows)")
print("\nCategory split summary:")
print(split_stats.sort_index())

Saved: ../Data/train.csv (20337 rows)
Saved: ../Data/test.csv (5084 rows)

Category split summary:
          train_rows  test_rows  total_rows  train_pct  test_pct
category                                                        
0              10734       2683       13417       80.0      20.0
1               9603       2401       12004       80.0      20.0
